ETAPA 1 — Modelos de tradução

1.1 RNN Tradutor

1.1 Tradutor com RNN/LSTM

As redes neurais recorrentes (RNN) são estruturas projetadas para lidar com dados sequenciais, ou seja, situações em que a ordem das informações influencia diretamente o resultado, como em textos e frases.

Entretanto, as RNNs tradicionais apresentam dificuldades ao trabalhar com dependências de longo prazo. Para contornar esse problema, surgiu a arquitetura LSTM (Long Short-Term Memory), que introduz mecanismos de controle de memória. Esses mecanismos permitem ao modelo decidir quais informações devem ser mantidas ao longo do tempo e quais podem ser descartadas, tornando o aprendizado mais eficiente em sequências maiores.

In [23]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [24]:


# Base pequena de exemplo
entrada_textos = [
    "hello",
    "good morning",
    "good night",
    "how are you",
    "thank you",
    "i love you",
    "see you later"
]

saida_textos = [
    "<start> olá <end>",
    "<start> bom dia <end>",
    "<start> boa noite <end>",
    "<start> como você está <end>",
    "<start> obrigado <end>",
    "<start> eu te amo <end>",
    "<start> até mais tarde <end>"
]

In [25]:
# Tokenização
tokenizer_entrada = Tokenizer()
tokenizer_saida = Tokenizer(filters='')

tokenizer_entrada.fit_on_texts(entrada_textos)
tokenizer_saida.fit_on_texts(saida_textos)

entrada_seq = tokenizer_entrada.texts_to_sequences(entrada_textos)
saida_seq = tokenizer_saida.texts_to_sequences(saida_textos)

max_entrada = max(len(seq) for seq in entrada_seq)
max_saida = max(len(seq) for seq in saida_seq)

entrada_seq = pad_sequences(entrada_seq, maxlen=max_entrada, padding="post")
saida_seq = pad_sequences(saida_seq, maxlen=max_saida, padding="post")

vocab_entrada = len(tokenizer_entrada.word_index) + 1
vocab_saida = len(tokenizer_saida.word_index) + 1

# Entrada e saída do decoder
decoder_input = saida_seq[:, :-1]
decoder_output = saida_seq[:, 1:]

In [26]:
# Modelo Encoder-Decoder com LSTM
latent_dim = 64

# Configuração do Encoder
encoder_inputs = Input(shape=(max_entrada,))
encoder_emb = Embedding(vocab_entrada, latent_dim)(encoder_inputs)
encoder_lstm, state_h, state_c = LSTM(latent_dim, return_state=True)(encoder_emb)

# Configuração do Decoder
decoder_inputs = Input(shape=(max_saida - 1,))
decoder_emb = Embedding(vocab_saida, latent_dim)(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True)(decoder_emb, initial_state=[state_h, state_c])
decoder_dense = Dense(vocab_saida, activation="softmax")(decoder_lstm)

# Definição e Compilação do Modelo
model = Model([encoder_inputs, decoder_inputs], decoder_dense)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Treinamento
model.fit(
    [entrada_seq, decoder_input],
    np.expand_dims(decoder_output, -1),
    epochs=300,
    verbose=0
)

print("Modelo RNN/LSTM treinado.")

Modelo RNN/LSTM treinado.


In [27]:
def traduzir(frase):
    entrada = tokenizer_entrada.texts_to_sequences([frase])
    entrada = pad_sequences(entrada, maxlen=max_entrada, padding="post")

    decoder_seq = tokenizer_saida.texts_to_sequences(["<start>"])[0]
    decoder_seq = pad_sequences([decoder_seq], maxlen=max_saida - 1, padding="post")

    pred = model.predict([entrada, decoder_seq], verbose=0)
    pred_indices = np.argmax(pred[0], axis=1)

    palavras = []
    index_word = tokenizer_saida.index_word

    for idx in pred_indices:
        palavra = index_word.get(idx, "")
        if palavra == "<end>":
            break
        if palavra not in ["<start>", ""]:
            palavras.append(palavra)

    return " ".join(palavras)

print(traduzir("good morning"))
print(traduzir("thank you"))
print(traduzir("good night"))
print(traduzir("world"))

bom dia
obrigado
boa noite
olá


In [28]:
print(traduzir("i love you"))

eu te amo


# 1.2 Transformer Tradutor



In [29]:
!pip install transformers sentencepiece -q

In [30]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-en-ROMANCE"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def traduzir(texto):
    inputs = tokenizer(texto, return_tensors="pt", padding=True)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(traduzir("We need your help Starfox!"))

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Necesitamos su ayuda Starfox!


In [31]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Modelo específico e robusto para tradução de Inglês para Português
model_name = "Helsinki-NLP/opus-mt-tc-big-en-pt"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def traduzir(texto):
    inputs = tokenizer(texto, return_tensors="pt", padding=True)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(traduzir("Never feared for anything"))
print(traduzir("Never shamed, but never free"))
print(traduzir("A life that healed the broken heart with all that it could"))
print(traduzir("Lived a life so endlessly"))
print(traduzir("Saw beyond what others see"))
print(traduzir("I tried to heal your broken heart with all that I could"))

print(traduzir("Will you stay?"))
print(traduzir("Will you stay away forever?"))

print(traduzir("How do I live without the ones I love?"))
print(traduzir("Time still turns the pages of the book it's burned"))
print(traduzir("Place and time always on my mind"))
print(traduzir("I have so much to say, but you're so far away"))

print(traduzir("Plans of what our futures hold"))
print(traduzir("Foolish lies of growing old"))
print(traduzir("It seems we're so invincible, the truth is so cold"))
print(traduzir("A final song, a last request"))
print(traduzir("A perfect chapter laid to rest"))
print(traduzir("Now and then I try to find a place in my mind"))

print(traduzir("Where you can stay"))
print(traduzir("You can stay awake forever"))

print(traduzir("How do I live without the ones I love?"))
print(traduzir("Time still turns the pages of the book it's burned"))
print(traduzir("Place and time always on my mind"))
print(traduzir("And the light you left remains, but it's so hard to stay"))
print(traduzir("When I have so much to say and you're so far away"))

print(traduzir("Sleep tight, I'm not afraid"))
print(traduzir("The ones that we love are here with me"))
print(traduzir("Lay away a place for me"))
print(traduzir("Cause as soon as I'm done, I'll be on my way"))
print(traduzir("To live eternally"))

print(traduzir("I love you, you were ready"))
print(traduzir("The pain is strong and urges rise"))
print(traduzir("But I'll see you when He lets me"))
print(traduzir("Your pain is gone, your hands untied"))

print(traduzir("So far away"))
print(traduzir("And I need you to know"))

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Nunca tem medo de nada
Nunca envergonhada, mas nunca livre
Uma vida que curou o coração partido com tudo o que podia
Viveu uma vida tão interminavelmente
Vi além do que os outros vêem
Tentei curar seu coração partido com tudo o que pude
Vais ficar?
Você vai ficar longe para sempre?
Como viver sem as pessoas que amo?
O tempo ainda vira as páginas do livro que está queimado
Lugar e tempo sempre na minha mente
Eu tenho tanto a dizer, mas você está tão longe
Planos do que o nosso futuro nos reserva
Mentiras mentirosas de envelhecer
Parece que somos tão invencíveis, a verdade é tão fria
Uma canção final, um último pedido
Um capítulo perfeito para descansar
De vez em quando eu tento encontrar um lugar na minha mente
Onde você pode ficar
Você pode ficar acordado para sempre
Como viver sem as pessoas que amo?
O tempo ainda vira as páginas do livro que está queimado
Lugar e tempo sempre na minha mente
E a luz que você deixou permanece, mas é tão difícil ficar
Quando eu tenho tanto a dizer e voc

A tradução gerada preserva parcialmente o sentido do trecho analisado, baseado na música So Far Away, da banda Avenged Sevenfold.

De modo geral, o modelo consegue transmitir a mensagem principal, porém apresenta limitações na fluidez e na adequação linguística. Um exemplo é “Never feared for anything”, traduzido como “Nunca tem medo de nada”, onde o tempo verbal não corresponde ao original, sendo mais natural “Nunca tive medo de nada”.

Outro caso é “Foolish lies of growing old”, traduzido como “Mentiras mentirosas de envelhecer”, evidenciando redundância e perda de sentido. Além disso, algumas frases foram traduzidas de forma muito literal, o que compromete a naturalidade e a interpretação do conteúdo.

Assim, conclui-se que o modelo mantém a ideia geral do texto, mas apresenta dificuldades com expressões mais complexas e construções menos diretas.

# ETAPA 2 — TF-IDF e análise sentimental com VADER

## 2.1 Modelo de classificação de termos com TF-IDF

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Criando os documentos
documentos = [
    "O chatbot é inteligente e responde bem.",
    "O chatbot usa Machine Learning para aprender.",
    "Machine Learning é importante para IA."
]

# Criando o modelo TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documentos)

# Mostrando os termos e suas pontuações
print(vectorizer.get_feature_names_out())  # Lista de palavras analisadas
print(tfidf_matrix.toarray())  # Matriz TF-IDF dos documentos

['aprender' 'bem' 'chatbot' 'ia' 'importante' 'inteligente' 'learning'
 'machine' 'para' 'responde' 'usa']
[[0.         0.52863461 0.40204024 0.         0.         0.52863461
  0.         0.         0.         0.52863461 0.        ]
 [0.48148213 0.         0.36617957 0.         0.         0.
  0.36617957 0.36617957 0.36617957 0.         0.48148213]
 [0.         0.         0.         0.51741994 0.51741994 0.
  0.3935112  0.3935112  0.3935112  0.         0.        ]]


Artigo 1:

ACHARYA, Anal; SINHA, Devadatta. Early prediction of students performance using machine learning techniques. International Journal of Computer Applications, v. 107, n. 1, p. 37-43, 2014.

Artigo 2:

WINKLER, Rainer; SÖLLNER, Matthias. Unleashing the potential of chatbots in education: A state-of-the-art analysis. In: Academy of management proceedings. Briarcliff Manor, NY 10510: Academy of Management, 2018. p. 15903.


In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Resumos adaptados dos artigos escolhidos
artigo1 = """
This study by Acharya and Sinha investigates the early prediction of student performance using machine learning techniques.
The authors analyze academic data to identify patterns that can indicate student success or failure in advance.
Different classification algorithms are applied to evaluate their effectiveness in predicting performance.
The results demonstrate that machine learning models can support educational institutions in decision-making and early intervention strategies.
"""

artigo2 = """
This work by Winkler and Söllner explores the use of chatbots in education through a state-of-the-art analysis.
The study discusses how chatbots can enhance learning experiences by providing support, interaction, and personalized assistance to students.
It also examines current challenges, limitations, and future opportunities of chatbot adoption in educational environments.
The findings highlight the potential of conversational agents as tools to improve engagement and learning outcomes.
"""

# Agrupando os textos
corpus = [artigo1, artigo2]

# Criando o modelo TF-IDF
modelo = TfidfVectorizer()

# Ajustando e transformando os dados
matriz = modelo.fit_transform(corpus)

# Exibindo resultados
print("Vocabulário identificado:")
print(modelo.get_feature_names_out())

print("\nMatriz TF-IDF:")
print(matriz.toarray())

Vocabulário identificado:
['academic' 'acharya' 'adoption' 'advance' 'agents' 'algorithms' 'also'
 'analysis' 'analyze' 'and' 'applied' 'are' 'art' 'as' 'assistance'
 'authors' 'by' 'can' 'challenges' 'chatbot' 'chatbots' 'classification'
 'conversational' 'current' 'data' 'decision' 'demonstrate' 'different'
 'discusses' 'early' 'education' 'educational' 'effectiveness'
 'engagement' 'enhance' 'environments' 'evaluate' 'examines' 'experiences'
 'explores' 'failure' 'findings' 'future' 'highlight' 'how' 'identify'
 'improve' 'in' 'indicate' 'institutions' 'interaction' 'intervention'
 'investigates' 'it' 'learning' 'limitations' 'machine' 'making' 'models'
 'of' 'opportunities' 'or' 'outcomes' 'patterns' 'performance'
 'personalized' 'potential' 'predicting' 'prediction' 'providing'
 'results' 'sinha' 'state' 'strategies' 'student' 'students' 'study'
 'success' 'support' 'söllner' 'techniques' 'that' 'the' 'their' 'this'
 'through' 'to' 'tools' 'use' 'using' 'winkler' 'work']

Matriz T

In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Resumos dos artigos
artigo1 = """
This study by Acharya and Sinha focuses on predicting student performance using machine learning techniques.
The research analyzes academic data to identify patterns that indicate student success or failure.
Different classification algorithms are applied to evaluate predictive accuracy.
The results show that machine learning can support early intervention and improve decision-making in education.
"""

artigo2 = """
This work by Winkler and Söllner analyzes the use of chatbots in education.
The study highlights how chatbots can improve interaction, provide personalized support, and enhance learning experiences.
It also discusses challenges and limitations related to chatbot adoption.
The results indicate strong potential for improving engagement and educational outcomes.
"""

# Lista de documentos
corpus = [artigo1, artigo2]

# Removendo palavras comuns (stopwords)
modelo_tfidf = TfidfVectorizer(stop_words='english')

# Aplicando TF-IDF
matriz = modelo_tfidf.fit_transform(corpus)

# Vocabulário
termos = modelo_tfidf.get_feature_names_out()
print("Vocabulário identificado:")
print(termos)

# Criando DataFrame
df_tfidf = pd.DataFrame(
    matriz.toarray(),
    columns=termos,
    index=["Artigo 1", "Artigo 2"]
)

print("\nMatriz TF-IDF:")
print(df_tfidf)

# Termos mais relevantes
for nome_artigo in df_tfidf.index:
    print(f"\nTermos mais relevantes do {nome_artigo}:")
    print(df_tfidf.loc[nome_artigo].sort_values(ascending=False).head(10))

Vocabulário identificado:
['academic' 'accuracy' 'acharya' 'adoption' 'algorithms' 'analyzes'
 'applied' 'challenges' 'chatbot' 'chatbots' 'classification' 'data'
 'decision' 'different' 'discusses' 'early' 'education' 'educational'
 'engagement' 'enhance' 'evaluate' 'experiences' 'failure' 'focuses'
 'highlights' 'identify' 'improve' 'improving' 'indicate' 'interaction'
 'intervention' 'learning' 'limitations' 'machine' 'making' 'outcomes'
 'patterns' 'performance' 'personalized' 'potential' 'predicting'
 'predictive' 'provide' 'related' 'research' 'results' 'sinha' 'strong'
 'student' 'study' 'success' 'support' 'söllner' 'techniques' 'use'
 'using' 'winkler' 'work']

Matriz TF-IDF:
          academic  accuracy   acharya  adoption  algorithms  analyzes  \
Artigo 1  0.161021  0.161021  0.161021  0.000000    0.161021  0.114568   
Artigo 2  0.000000  0.000000  0.000000  0.182422    0.000000  0.129795   

           applied  challenges   chatbot  chatbots  ...   student     study  \
Arti

Os artigos apresentam termos em comum relacionados ao uso de tecnologias aplicadas à educação, como learning, models, data e analysis, indicando que ambos estão inseridos no contexto de aplicação de inteligência artificial no ensino.
A partir da análise dos resultados, observa-se que o Artigo 1 possui um foco mais técnico, enfatizando o uso de algoritmos de machine learning para prever o desempenho dos estudantes. Já o Artigo 2 apresenta uma abordagem mais aplicada, destacando o uso de chatbots como ferramenta de apoio ao processo educacional.
Dessa forma, conclui-se que, embora os dois trabalhos estejam inseridos no mesmo domínio, eles se diferenciam principalmente no objetivo: enquanto um busca prever resultados acadêmicos por meio de modelos preditivos, o outro explora o uso de sistemas interativos para melhorar a experiência de aprendizagem.

## 2.2 Modelo de análise sentimental com VADER

Vídeo: https://youtu.be/jNQXAC9IVRw

In [35]:
!pip install vaderSentiment

# IMPORTAÇÃO DAS BIBLIOTECAS
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# INICIALIZAÇÃO DO ANALISADOR DE SENTIMENTO
analisador = SentimentIntensityAnalyzer()

# FUNÇÃO DE ANÁLISE DE SENTIMENTO
def analisar_sentimento_vader(texto):

    sentimento = analisador.polarity_scores(texto)

    if sentimento['compound'] >= 0.05:
        return "Positivo", sentimento
    elif sentimento['compound'] <= -0.05:
        return "Negativo", sentimento
    else:
        return "Neutro", sentimento

# COMENTÁRIOS SELECIONADOS
mensagem1 = "We're so honored that the first ever YouTube video was filmed here!"
mensagem2 = "This started it all...... bruh 😳"
mensagem3 = "If he uploads another video with the title: Me leaving the zoo, that will be the end of YouTube"
mensagem4 = "Hey 21 years later!"
mensagem5 = "Honestly so glad to hear that removing dislikes is stupid."

# EXECUÇÃO
for i, msg in enumerate([mensagem1, mensagem2, mensagem3, mensagem4, mensagem5], start=1):
    classificacao, score = analisar_sentimento_vader(msg)

    print(f"\nComentário {i}:")
    print(f"Texto: {msg}")
    print(f"Classificação: {classificacao}")
    print(f"Score: {score}")


Comentário 1:
Texto: We're so honored that the first ever YouTube video was filmed here!
Classificação: Positivo
Score: {'neg': 0.0, 'neu': 0.715, 'pos': 0.285, 'compound': 0.6581}

Comentário 2:
Texto: This started it all...... bruh 😳
Classificação: Neutro
Score: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}

Comentário 3:
Texto: If he uploads another video with the title: Me leaving the zoo, that will be the end of YouTube
Classificação: Neutro
Score: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}

Comentário 4:
Texto: Hey 21 years later!
Classificação: Neutro
Score: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}

Comentário 5:
Texto: Honestly so glad to hear that removing dislikes is stupid.
Classificação: Neutro
Score: {'neg': 0.332, 'neu': 0.326, 'pos': 0.342, 'compound': 0.0498}


### Análise dos resultados com VADER

**O modelo VADER classificou corretamente os comentários? Houve limitações relacionadas ao idioma, ironia, contexto ou tamanho do texto?**

---

**1 - We're so honored that the first ever YouTube video was filmed here!**

O VADER classificou corretamente como positivo (`compound: 0.6581`). O resultado faz sentido, pois o comentário contém palavras claramente positivas, como *“honored”*, facilitando a identificação do sentimento.

---

**2 - This started it all...... bruh 😳**

O VADER não classificou corretamente. O resultado foi neutro (`compound: 0.0`), porém o comentário possui um tom positivo ou de admiração. A presença de gírias (*“bruh”*) e emoji não foi interpretada pelo modelo, evidenciando limitação no reconhecimento de linguagem informal.

---

**3 - If he uploads another video with the title: Me leaving the zoo, that will be the end of YouTube**

O VADER não classificou corretamente. Apesar de ter um tom emocional (misto entre humor e possível crítica), o modelo retornou neutro (`compound: 0.0`). Isso ocorre porque não há palavras com polaridade clara, mostrando dificuldade na interpretação de ironia e contexto.

---

**4 - Hey 21 years later!**

O VADER classificou como neutro (`compound: 0.0`), o que está correto do ponto de vista literal. No entanto, o comentário carrega um sentimento implícito de surpresa ou nostalgia, que não é captado pelo modelo.

---

**5 - Honestly so glad to hear that removing dislikes is stupid.**

O VADER apresentou uma limitação clara. O resultado foi classificado como neutro (`compound: 0.0498`), porém o comentário possui uma crítica explícita (*“stupid”*). A presença simultânea de termos positivos (*“glad”*) e negativos (*“stupid”*) gerou ambiguidade, levando o modelo a um resultado próximo do neutro.

---

### Conclusão geral

O modelo VADER demonstrou bom desempenho na identificação de sentimentos explícitos, especialmente quando há palavras claramente positivas ou negativas. Entretanto, apresentou limitações relevantes na interpretação de linguagem informal, gírias, emojis, ironia e sentimentos implícitos. Além disso, comentários com polaridade mista tendem a gerar classificações imprecisas. Dessa forma, embora seja eficiente em cenários simples, o modelo pode não representar adequadamente o sentimento em textos mais complexos ou contextuais.

# ETAPA 3 — Análise sentimental com modelos Transformers

## 3.1 Modelo sentimental com RoBERTa

In [36]:
from transformers import pipeline

# Inicializando pipeline com modelo RoBERTa multilíngue
modelo_sentimento = pipeline(
    task="sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment"
)

# Comentários baseados no vídeo "Me at the zoo"
comentarios = [
    "Incrível pensar que esse foi o primeiro vídeo do YouTube!",
    "Esse vídeo começou tudo, impressionante 😳",
    "Se ele postar outro vídeo saindo do zoológico, vai ser histórico",
    "Mais de 20 anos depois e ainda estamos assistindo isso",
    "Eu gosto muito desse vídeo, mas a decisão de remover dislikes foi péssima"
]

# Aplicando análise de sentimento
for i, texto in enumerate(comentarios, start=1):
    resultado = modelo_sentimento(texto)

    print(f"\nComentário {i}:")
    print(f"Texto: {texto}")
    print(f"Resultado: {resultado}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Comentário 1:
Texto: Incrível pensar que esse foi o primeiro vídeo do YouTube!
Resultado: [{'label': 'positive', 'score': 0.8203388452529907}]

Comentário 2:
Texto: Esse vídeo começou tudo, impressionante 😳
Resultado: [{'label': 'positive', 'score': 0.5737176537513733}]

Comentário 3:
Texto: Se ele postar outro vídeo saindo do zoológico, vai ser histórico
Resultado: [{'label': 'positive', 'score': 0.4226570129394531}]

Comentário 4:
Texto: Mais de 20 anos depois e ainda estamos assistindo isso
Resultado: [{'label': 'neutral', 'score': 0.5089981555938721}]

Comentário 5:
Texto: Eu gosto muito desse vídeo, mas a decisão de remover dislikes foi péssima
Resultado: [{'label': 'negative', 'score': 0.9425840973854065}]


### Análise dos resultados com RoBERTa

**O modelo RoBERTa classificou corretamente os comentários? Houve limitações relacionadas ao contexto, ironia ou ambiguidade?**

---

**1 - Incrível pensar que esse foi o primeiro vídeo do YouTube!**

O RoBERTa classificou corretamente como positivo (`score: 0.82`). O comentário possui uma expressão clara de admiração (*“incrível”*), o que facilita a identificação do sentimento.

---

**2 - Esse vídeo começou tudo, impressionante 😳**

O RoBERTa classificou corretamente como positivo (`score: 0.57`). Apesar do uso de emoji, o modelo conseguiu capturar o tom de surpresa e admiração presente no comentário.

---

**3 - Se ele postar outro vídeo saindo do zoológico, vai ser histórico**

O RoBERTa classificou como positivo (`score: 0.42`). Embora o score seja mais baixo, a classificação faz sentido, pois o comentário expressa expectativa e valorização do evento.

---

**4 - Mais de 20 anos depois e ainda estamos assistindo isso**

O RoBERTa classificou como neutro (`score: 0.50`). A classificação é aceitável, pois o comentário é mais descritivo. No entanto, pode haver um sentimento implícito de nostalgia, que não foi totalmente captado pelo modelo.

---

**5 - Eu gosto muito desse vídeo, mas a decisão de remover dislikes foi péssima**

O RoBERTa classificou corretamente como negativo (`score: 0.94`). Apesar de iniciar com um tom positivo, o comentário apresenta uma crítica forte (*“péssima”*), que foi corretamente identificada como predominante.

---

### Conclusão geral

O modelo RoBERTa apresentou bom desempenho na maioria dos casos, conseguindo identificar corretamente sentimentos positivos e negativos, mesmo em frases com maior complexidade. Em comparação com abordagens mais simples, como o VADER, o modelo demonstrou maior capacidade de interpretar contexto e lidar com linguagem natural.

Entretanto, ainda apresenta limitações na identificação de sentimentos implícitos, como nostalgia, e em casos onde o texto não possui palavras explicitamente emocionais. Ainda assim, mostrou-se mais robusto e consistente na análise de sentimentos.

## 3.2 BERT/BERTimbau

In [37]:
from transformers import pipeline

# Inicializando pipeline com modelo BERT multilíngue
modelo_bert = pipeline(
    task="sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

# Comentários baseados no vídeo "Me at the zoo"
comentarios = [
    "Incrível pensar que esse foi o primeiro vídeo do YouTube!",
    "Esse vídeo começou tudo, impressionante 😳",
    "Se ele postar outro vídeo saindo do zoológico, vai ser histórico",
    "Mais de 20 anos depois e ainda estamos assistindo isso",
    "Eu gosto muito desse vídeo, mas a decisão de remover dislikes foi péssima"
]

# Aplicando análise de sentimento
for i, texto in enumerate(comentarios, start=1):
    resultado = modelo_bert(texto)

    print(f"\nComentário {i}:")
    print(f"Texto: {texto}")
    print(f"Resultado: {resultado}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Comentário 1:
Texto: Incrível pensar que esse foi o primeiro vídeo do YouTube!
Resultado: [{'label': '1 star', 'score': 0.4471787214279175}]

Comentário 2:
Texto: Esse vídeo começou tudo, impressionante 😳
Resultado: [{'label': '5 stars', 'score': 0.7643229365348816}]

Comentário 3:
Texto: Se ele postar outro vídeo saindo do zoológico, vai ser histórico
Resultado: [{'label': '3 stars', 'score': 0.2867778539657593}]

Comentário 4:
Texto: Mais de 20 anos depois e ainda estamos assistindo isso
Resultado: [{'label': '5 stars', 'score': 0.6260523796081543}]

Comentário 5:
Texto: Eu gosto muito desse vídeo, mas a decisão de remover dislikes foi péssima
Resultado: [{'label': '2 stars', 'score': 0.40186089277267456}]


### Análise dos resultados com BERT (multilíngue)

**O modelo BERT classificou corretamente os comentários? Houve limitações relacionadas ao contexto, ironia ou ambiguidade?**

---

**1 - Incrível pensar que esse foi o primeiro vídeo do YouTube!**

O modelo BERT classificou corretamente como positivo (alta pontuação em estrelas). O comentário possui uma expressão clara de admiração, o que facilita a identificação do sentimento.

---

**2 - Esse vídeo começou tudo, impressionante 😳**

O modelo BERT classificou corretamente como positivo. Mesmo com o uso de emoji, o modelo conseguiu interpretar o tom de surpresa e reconhecimento da importância do vídeo.

---

**3 - Se ele postar outro vídeo saindo do zoológico, vai ser histórico**

O modelo BERT classificou corretamente como positivo, porém com score mais baixo. Isso ocorre porque o sentimento é mais implícito (expectativa), e não uma emoção explícita.

---

**4 - Mais de 20 anos depois e ainda estamos assistindo isso**

O modelo classificou como neutro ou levemente positivo. A classificação é aceitável, pois o comentário é mais descritivo, mas existe um sentimento implícito de nostalgia que não é totalmente captado.

---

**5 - Eu gosto muito desse vídeo, mas a decisão de remover dislikes foi péssima**

O modelo BERT pode apresentar inconsistência nesse caso. Apesar da crítica forte (*“péssima”*), a presença de um início positivo (*“gosto muito”*) pode influenciar a classificação, gerando um resultado menos preciso dependendo do score retornado.

---

### Conclusão geral

O modelo BERT apresentou bom desempenho na maioria dos casos, conseguindo identificar corretamente sentimentos mais explícitos. No entanto, assim como outros modelos, ainda apresenta limitações na interpretação de sentimentos implícitos, ironia e comentários com polaridade mista.

De modo geral, mostrou-se consistente, mas pode gerar interpretações menos precisas em textos mais subjetivos ou ambíguos.

## Comparativo

Comparando os dois modelos de análise de sentimentos, observa-se que ambos apresentaram bons resultados em comentários com sentimento mais explícito. Exemplos disso são frases como "Incrível pensar que esse foi o primeiro vídeo do YouTube!" e "Esse vídeo começou tudo, impressionante", onde tanto o modelo RoBERTa quanto o modelo BERT conseguiram identificar corretamente o tom positivo.

Por outro lado, os modelos apresentaram dificuldades em comentários com linguagem mais informal ou com uso de gírias e emojis. Um exemplo é o comentário "This started it all...... bruh 😳", no qual, apesar do tom de admiração, os modelos podem ter dificuldade em interpretar corretamente devido à presença de linguagem informal e elementos não textuais.

Também foram observadas diferenças no tratamento de comentários com maior ambiguidade. No caso do comentário "Se ele postar outro vídeo saindo do zoológico, vai ser histórico", o sentimento é mais implícito, baseado em expectativa. O RoBERTa conseguiu capturar melhor esse contexto, enquanto o BERT apresentou menor confiança na classificação.

Já no comentário "Mais de 20 anos depois e ainda estamos assistindo isso", ambos os modelos tenderam a classificá-lo como neutro. No entanto, a frase pode carregar um sentimento implícito de nostalgia, que não é totalmente captado pelos modelos.

Por fim, no comentário "Eu gosto muito desse vídeo, mas a decisão de remover dislikes foi péssima", ambos os modelos demonstraram desafios ao lidar com polaridade mista. A presença de termos positivos e negativos na mesma frase pode gerar classificações menos precisas ou com menor confiança.

Assim, conclui-se que os dois modelos são eficazes na identificação de sentimentos explícitos, mas apresentam limitações em casos que envolvem linguagem informal, ambiguidade ou sentimentos implícitos. Dessa forma, a interpretação humana continua sendo essencial para complementar a análise automática.